# Question 2: Resampling and Frequency Conversion

This question focuses on resampling operations and frequency conversion using ICU monitoring data (hourly) and patient vital signs data (daily).

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('default')
sns.set_style('whitegrid')

# Create output directory
os.makedirs('output', exist_ok=True)

## Part 2.1: Load and Prepare Data

**Note:** These datasets have realistic characteristics:
- **ICU Monitoring**: 75 patients with variable stay lengths (2-30 days). Not all patients are present for the entire 6-month period - patients are admitted and discharged at different times.
- **Patient Vitals**: Already contains some missing visits (~5% missing data). This is realistic and will be useful for practicing missing data handling.

In [2]:
# Load ICU monitoring data (hourly)
icu_monitoring = pd.read_csv('data/icu_monitoring.csv')

# Load patient vitals data (daily) - for comparison
patient_vitals = pd.read_csv('data/patient_vitals.csv')

print("ICU monitoring shape:", icu_monitoring.shape)
print("Patient vitals shape:", patient_vitals.shape)

# Convert datetime columns and set as index
icu_monitoring['datetime'] = pd.to_datetime(icu_monitoring['datetime'])
icu_monitoring = icu_monitoring.set_index('datetime')

patient_vitals['date'] = pd.to_datetime(patient_vitals['date'])
patient_vitals = patient_vitals.set_index('date')

print("\nICU monitoring sample:")
print(icu_monitoring.head())
print("\nPatient vitals sample:")
print(patient_vitals.head())

# Check data characteristics
print(f"\nICU patients: {icu_monitoring['patient_id'].nunique()}")
print(f"ICU date range: {icu_monitoring.index.min()} to {icu_monitoring.index.max()}")
print(f"\nPatient vitals patients: {patient_vitals['patient_id'].nunique()}")
print(f"Patient vitals date range: {patient_vitals.index.min()} to {patient_vitals.index.max()}")

ICU monitoring shape: (86400, 7)
Patient vitals shape: (18250, 7)

ICU monitoring sample:
                    patient_id  heart_rate  blood_pressure_systolic  \
datetime                                                              
2023-01-01 00:00:00     ICU001   82.000000                      126   
2023-01-01 01:00:00     ICU001   98.294095                      128   
2023-01-01 02:00:00     ICU001  103.500000                      129   
2023-01-01 03:00:00     ICU001   91.535534                      136   
2023-01-01 04:00:00     ICU001   87.330127                      129   

                     blood_pressure_diastolic  oxygen_saturation  temperature  
datetime                                                                       
2023-01-01 00:00:00                        65                 96    98.783988  
2023-01-01 01:00:00                        67                 95    99.186212  
2023-01-01 02:00:00                        68                 94    98.800638  
2023-01-01 0

## Part 2.2: Time Series Selection

**⚠️ WARNING: Sort Index Before Date Selection!**
Since multiple patients share the same date, the `patient_vitals` index is non-monotonic (not strictly increasing). **You MUST sort the index first** before using `.loc` with date ranges:

In [3]:
patient_vitals = patient_vitals.sort_index()

Without sorting, pandas cannot reliably handle date range selections and may return unexpected results or errors.

**TODO: Perform time series indexing and selection**

In [4]:
import pandas as pd
import numpy as np
import os

os.makedirs('output', exist_ok=True)

icu_monitoring = pd.read_csv('data/icu_monitoring.csv')
patient_vitals = pd.read_csv('data/patient_vitals.csv')

# Ensure datetime columns are datetime type and set as index
icu_monitoring['datetime'] = pd.to_datetime(icu_monitoring['datetime'])
icu_monitoring = icu_monitoring.set_index('datetime')

patient_vitals['date'] = pd.to_datetime(patient_vitals['date'])
patient_vitals = patient_vitals.set_index('date')

numeric_cols_icu = icu_monitoring.select_dtypes(include=[np.number]).columns
icu_daily = icu_monitoring[numeric_cols_icu].resample('D').mean()
print("ICU daily shape:", icu_daily.shape)

# Resample with multiple aggregations
icu_daily_stats = icu_monitoring[numeric_cols_icu].resample('D').agg({
    'heart_rate': ['mean', 'max', 'min'],
    'temperature': 'mean'
})
print("ICU daily stats sample:")
print(icu_daily_stats.head())

# Resample patient vitals daily → weekly/monthly
numeric_cols_pv = patient_vitals.select_dtypes(include=[np.number]).columns

# Weekly
patient_vitals_weekly = patient_vitals[numeric_cols_pv].resample('W').mean()
print("Weekly resampled shape:", patient_vitals_weekly.shape)

# Monthly (Month End frequency)
patient_vitals_monthly = patient_vitals[numeric_cols_pv].resample('M').mean()
print("Monthly resampled shape:", patient_vitals_monthly.shape)

# Handle missing values
monthly_to_daily = patient_vitals_monthly.resample('D').asfreq()
print("Missing values after upsampling from monthly to daily:")
print(monthly_to_daily.isna().sum())

# Aggregate patient vitals by date for comparison
patient_vitals_reset = patient_vitals[numeric_cols_pv].reset_index()
patient_vitals_daily_agg = patient_vitals_reset.groupby('date').mean()
print("Aggregated daily patient vitals shape:", patient_vitals_daily_agg.shape)

# Compare resampling frequencies
resampling_comparison = pd.DataFrame({
    'frequency': ['daily', 'weekly', 'monthly'],
    'date_range': [
        f"{patient_vitals_daily_agg.index.min().date()} to {patient_vitals_daily_agg.index.max().date()}",
        f"{patient_vitals_weekly.index.min().date()} to {patient_vitals_weekly.index.max().date()}",
        f"{patient_vitals_monthly.index.min().date()} to {patient_vitals_monthly.index.max().date()}"
    ],
    'row_count': [
        len(patient_vitals_daily_agg),
        len(patient_vitals_weekly),
        len(patient_vitals_monthly)
    ],
    'mean_temperature': [
        patient_vitals_daily_agg['temperature'].mean(),
        patient_vitals_weekly['temperature'].mean(),
        patient_vitals_monthly['temperature'].mean()
    ],
    'std_temperature': [
        patient_vitals_daily_agg['temperature'].std(),
        patient_vitals_weekly['temperature'].std(),
        patient_vitals_monthly['temperature'].std()
    ]
})

print("\nResampling comparison:")
print(resampling_comparison)

resampling_comparison.to_csv('output/q2_resampling_analysis.csv', index=False)
print("\nResampling analysis saved to 'output/q2_resampling_analysis.csv'")


ICU daily shape: (180, 5)
ICU daily stats sample:
           heart_rate                   temperature
                 mean         max   min        mean
datetime                                           
2023-01-01  81.793729  111.829629  50.0   98.528348
2023-01-02  81.479854  108.829629  50.0   98.498305
2023-01-03  81.767332  110.330127  50.0   98.534337
2023-01-04  81.852771  110.000000  50.0   98.542113
2023-01-05  81.730187  109.829629  50.0   98.536312
Weekly resampled shape: (53, 5)
Monthly resampled shape: (12, 5)
Missing values after upsampling from monthly to daily:
temperature                 323
heart_rate                  323
blood_pressure_systolic     323
blood_pressure_diastolic    323
weight                      323
dtype: int64
Aggregated daily patient vitals shape: (365, 5)

Resampling comparison:
  frequency                date_range  row_count  mean_temperature  \
0     daily  2023-01-01 to 2023-12-31        365         98.660538   
1    weekly  2023-01-01 to 20

/var/folders/c0/4jn3wz693dbg7vh1dl8yg06r0000gp/T/ipykernel_16314/4006226425.py:37: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  patient_vitals_monthly = patient_vitals[numeric_cols_pv].resample('M').mean()


## Part 2.3: Resampling Operations

**TODO: Perform resampling and frequency conversion**

**Important Note:** When resampling DataFrames that contain non-numeric columns (like `patient_id`), you'll get an error if you try to aggregate them with numeric functions like `mean()`. Use `df.select_dtypes(include=[np.number])` to select only numeric columns before resampling, or specify which columns to aggregate in `.agg()`.

In [6]:
import pandas as pd
import numpy as np
import os

# Ensure output directory exists
os.makedirs('output', exist_ok=True)


# Load data
icu_monitoring = pd.read_csv('data/icu_monitoring.csv')
patient_vitals = pd.read_csv('data/patient_vitals.csv')

# Convert datetime columns and set index
icu_monitoring['datetime'] = pd.to_datetime(icu_monitoring['datetime'])
icu_monitoring = icu_monitoring.set_index('datetime')

patient_vitals['date'] = pd.to_datetime(patient_vitals['date'])
patient_vitals = patient_vitals.set_index('date')


# Resample ICU hourly → daily
numeric_cols_icu = icu_monitoring.select_dtypes(include=[np.number]).columns
icu_daily = icu_monitoring[numeric_cols_icu].resample('D').mean()
print("ICU daily shape:", icu_daily.shape)

# Resample with multiple aggregations
icu_daily_stats = icu_monitoring[numeric_cols_icu].resample('D').agg({
    'heart_rate': ['mean', 'max', 'min'],
    'temperature': 'mean'
})
print("ICU daily stats sample:")
print(icu_daily_stats.head())

# Resample patient vitals daily → weekly/monthly
numeric_cols_pv = patient_vitals.select_dtypes(include=[np.number]).columns

# Weekly
patient_vitals_weekly = patient_vitals[numeric_cols_pv].resample('W').mean()
print("Weekly resampled shape:", patient_vitals_weekly.shape)

# Monthly (Month End frequency)
patient_vitals_monthly = patient_vitals[numeric_cols_pv].resample('M').mean()
print("Monthly resampled shape:", patient_vitals_monthly.shape)


# Upsample monthly → daily (create missing values)

monthly_to_daily = patient_vitals_monthly.resample('D').asfreq()
print("Missing values after upsampling from monthly to daily:")
print(monthly_to_daily.isna().sum())


# Aggregate patient vitals by date for comparison
patient_vitals_reset = patient_vitals[numeric_cols_pv].reset_index()
patient_vitals_daily_agg = patient_vitals_reset.groupby('date').mean()
print("Aggregated daily patient vitals shape:", patient_vitals_daily_agg.shape)


# Compare resampling frequencies

resampling_comparison = pd.DataFrame({
    'frequency': ['daily', 'weekly', 'monthly'],
    'date_range': [
        f"{patient_vitals_daily_agg.index.min().date()} to {patient_vitals_daily_agg.index.max().date()}",
        f"{patient_vitals_weekly.index.min().date()} to {patient_vitals_weekly.index.max().date()}",
        f"{patient_vitals_monthly.index.min().date()} to {patient_vitals_monthly.index.max().date()}"
    ],
    'row_count': [
        len(patient_vitals_daily_agg),
        len(patient_vitals_weekly),
        len(patient_vitals_monthly)
    ],
    'mean_temperature': [
        patient_vitals_daily_agg['temperature'].mean(),
        patient_vitals_weekly['temperature'].mean(),
        patient_vitals_monthly['temperature'].mean()
    ],
    'std_temperature': [
        patient_vitals_daily_agg['temperature'].std(),
        patient_vitals_weekly['temperature'].std(),
        patient_vitals_monthly['temperature'].std()
    ]
})

print("\nResampling comparison:")
print(resampling_comparison)


# Save results

resampling_comparison.to_csv('output/q2_resampling_analysis.csv', index=False)
print("\nResampling analysis saved to 'output/q2_resampling_analysis.csv'")


ICU daily shape: (180, 5)
ICU daily stats sample:
           heart_rate                   temperature
                 mean         max   min        mean
datetime                                           
2023-01-01  81.793729  111.829629  50.0   98.528348
2023-01-02  81.479854  108.829629  50.0   98.498305
2023-01-03  81.767332  110.330127  50.0   98.534337
2023-01-04  81.852771  110.000000  50.0   98.542113
2023-01-05  81.730187  109.829629  50.0   98.536312
Weekly resampled shape: (53, 5)
Monthly resampled shape: (12, 5)
Missing values after upsampling from monthly to daily:
temperature                 323
heart_rate                  323
blood_pressure_systolic     323
blood_pressure_diastolic    323
weight                      323
dtype: int64
Aggregated daily patient vitals shape: (365, 5)

Resampling comparison:
  frequency                date_range  row_count  mean_temperature  \
0     daily  2023-01-01 to 2023-12-31        365         98.660538   
1    weekly  2023-01-01 to 20

/var/folders/c0/4jn3wz693dbg7vh1dl8yg06r0000gp/T/ipykernel_16314/4017616721.py:42: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  patient_vitals_monthly = patient_vitals[numeric_cols_pv].resample('M').mean()


## Part 2.4: Missing Data Handling

**💡 TIP: High Percentage of Missing Data is Expected!**
When upsampling from monthly to daily frequency, you'll create approximately 96% missing data (only 12 month-end dates have values out of 365 days). This is normal and expected for upsampling - don't be alarmed!

**Approach:** Create missing values by upsampling monthly data to daily frequency. This creates a clear, structured pattern of missing data that's ideal for practicing imputation methods.

**TODO: Handle missing data in time series**

In [8]:
import pandas as pd
import numpy as np
import os

os.makedirs('output', exist_ok=True)

# ----------------------------
# 0. Prepare monthly patient vitals data (assumes patient_vitals_monthly exists)
# ----------------------------
# Upsample monthly → daily to create missing values (~96%)
ts_with_missing = patient_vitals_monthly['temperature'].resample('D').asfreq()
print("Missing value count:", ts_with_missing.isna().sum())
print("Missing value percentage:", ts_with_missing.isna().sum() / len(ts_with_missing) * 100)

# ----------------------------
# 1. Impute missing values
# ----------------------------
ts_ffill = ts_with_missing.ffill()
ts_bfill = ts_with_missing.bfill()
ts_interpolated_linear = ts_with_missing.interpolate(method='linear')
ts_interpolated_time = ts_with_missing.interpolate(method='time')
ts_rolling_imputed = ts_with_missing.fillna(ts_with_missing.rolling(3, min_periods=1).mean())

# ----------------------------
# 2. Missing data patterns
# ----------------------------
missing_by_month = ts_with_missing.groupby(ts_with_missing.index.month).apply(lambda x: x.isna().sum())
missing_by_day = ts_with_missing.groupby(ts_with_missing.index.dayofweek).apply(lambda x: x.isna().sum())
missing_patterns = f"Missing by month:\n{missing_by_month}\n\nMissing by day of week (Mon=0):\n{missing_by_day}"

# ----------------------------
# 3. Create missing data report
# ----------------------------
missing_data_report = f"""
Missing Data Handling Report

1. Missing value summary:
Total missing values: {ts_with_missing.isna().sum()}
Percentage missing: {ts_with_missing.isna().sum() / len(ts_with_missing) * 100:.2f}%

2. Missing data patterns:
{missing_patterns}

3. Imputation methods applied:
- Forward fill (ts_ffill)
- Backward fill (ts_bfill)
- Linear interpolation (ts_interpolated_linear)
- Time-based interpolation (ts_interpolated_time)
- Rolling mean imputation with 3-day window (ts_rolling_imputed)

4. Rationale:
Forward/backward fill maintains last known value and is simple for short gaps.
Interpolation estimates missing values based on surrounding known points.
Rolling mean smooths out fluctuations and reduces noise in imputed values.

5. Pros and cons:
- Forward/backward fill: Simple but can propagate errors if gaps are long.
- Interpolation: Captures trends but may underestimate variability.
- Rolling mean: Smooths but introduces lag.

6. Examples:
Original missing data (first 10 days of January):
{ts_with_missing.head(10)}

Forward fill imputed values:
{ts_ffill.head(10)}

Linear interpolation imputed values:
{ts_interpolated_linear.head(10)}
"""

# Save missing data report
with open('output/q2_missing_data_report.txt', 'w') as f:
    f.write(missing_data_report)

print("Missing data report saved to 'output/q2_missing_data_report.txt'")


Missing value count: 323
Missing value percentage: 96.41791044776119
Missing data report saved to 'output/q2_missing_data_report.txt'


## Submission Checklist

Before moving to Question 3, verify you've created:

- [ ] `output/q2_resampling_analysis.csv` - resampling analysis results
- [ ] `output/q2_missing_data_report.txt` - missing data handling report
